In [1]:
'''
Scripts for data analysis of JWST/NIRSpec IFU data
developed by I. Pascucci 2022-2025
several utilized in Pascucci et al. 2025NatAs...9...81
https://ui.adsabs.harvard.edu/abs/2025NatAs...9...81P/abstract
'''
import copy
import numpy as np

#matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.gridspec import GridSpec

#scipy
from scipy.ndimage import rotate
from scipy.optimize import curve_fit
from scipy.interpolate import CubicSpline
from scipy.ndimage import gaussian_filter1d
from scipy.signal import medfilt,savgol_filter
from scipy.signal import find_peaks, peak_widths

#sklearn
from sklearn import linear_model
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error

#astropy
import astropy.units as u
from astropy.wcs import WCS
from astropy.visualization import make_lupton_rgb
from astropy.io import fits
from astropy.wcs.utils import pixel_to_skycoord

#
import pdb
#

In [2]:
# Strip the header of a cube to a 2D to generate an RGB image
def fix_FITS(hdu):
    # hdu is the header of the datacube
    del hdu['NAXIS3']
    del hdu['WCSAXES']
    del hdu['CRPIX3']
    del hdu['CRVAL3']
    del hdu['CTYPE3']
    del hdu['CUNIT3']
    del hdu['CDELT3']
    del hdu['PC1_3']
    del hdu['PC2_3']
    del hdu['PC3_3']
    del hdu['PC3_1']
    del hdu['PC3_2']
    # Update the NAXIS and WCSAXES keywords to 2
    hdu['NAXIS'] = 2
    hdu['WCSAXES'] = 2
    return hdu

In [3]:
# Define a Gaussian function
def Gaussian(x, amp, cen, sig, cont_base=None, cont_slope=None):
    if (cont_base is None) & (cont_slope is None):
        Gauss2return = amp * np.exp(-0.5*((x-cen)/sig)**2)
    if (cont_base is not None) & (cont_slope is None):
        Gauss2return = amp * np.exp(-0.5*((x-cen)/sig)**2) + cont_base
    if (cont_base is not None) & (cont_slope is not None):
        Gauss2return = amp * np.exp(-0.5*((x-cen)/sig)**2) + cont_base + cont_slope*x
    return Gauss2return

In [4]:
def lmfitGaussian(x, amp, cen, sig, cont_base, cont_slope):
    Gauss2return = amp * np.exp(-0.5*((x-cen)/sig)**2) + cont_base + cont_slope*x
    return Gauss2return

In [5]:
#
# Quick line check: given an isolated line it estimates if the line might be detected
# Returns 1 if line might be detected, 0 otherwise
def quick_line_ck(xwave,yflux,xline):
    # Criteria for the possible existence of a line:
    # - peak emission > 3 times stddev [computed outside line, they are 5 pix wide]
    # - loc peak emission is within xline+/-NIRspec resolution
    xres = np.nanmean(np.diff(xwave))
    ymaxflux, idx_ymaxflux = np.nanmax(flux), np.nanargmax(flux)
    idx_xline = np.argmin(np.abs(xwave-xline))
    mask = np.ones_like(yflux, dtype=bool)
    mask[max(0, idx_xline - 2):idx_xline + 2 + 1] = False
    ystdflux = np.std(yflux[mask])
    if (ymaxflux > 3*ystdflux) and (xwave[idx_ymaxflux] <= xline+xres) and (xwave[idx_ymaxflux] >= xline-xres):
        quick_ck = 1 # Line is detected
    else:
        quick_ck = 0
    return quick_ck, xres

In [6]:
#
# Establish which of the lines listed in lines_to_search is detected (1)
# and assign a starting max_amplitude
def find_lines(xwave,yflux,lines_to_search,contaminating_lines=None,remove_duplicate='False'):
    # Criteria for the possible existence of a line within a subregion of 5 waves:
    # - loc peak emission is within line_to_search+/-NIRspec resolution and NOT contaminating_line+/- resolution
    # – peak emission >  1.1*min in those waves
    # By default all lines are assigned a disposition == 0 (not detection)
    # A line of interest is assigned a disposition == 1
    print('>>> Finding lines and remove duplicates...')
    xres = np.nanmean(np.diff(xwave))
    data = {
        'line': lines_to_search['line_wave_um'],
        'species': lines_to_search['species'],
        'disposition': [0] * len(lines_to_search['line_wave_um']),
        'max_amplitude': [None] * len(lines_to_search['line_wave_um']),
        'line_flux': [None] * len(lines_to_search['line_wave_um'])
        } # at start all lines are assumed to be non-detections and assigned NAN for max_amplitude
    lines_dispositions = pd.DataFrame(data)
    potential_lines = []
    for ii in range(len(lines_dispositions['line'])):
        cline  = lines_dispositions['line'][ii] # current line
        idx_cline = np.argmin(np.abs(xwave-cline)) # index of the wave closest to the line
        cwave, cflux = xwave[idx_cline-2:idx_cline+3], yflux[idx_cline-2:idx_cline+3]
        cflux_min, cflux_max, cflux_idx_max = np.nanmin(cflux), np.nanmax(cflux), np.argmax(cflux)
        #print(cflux_max/cflux_min, xwave[cflux_idx_max], cline-xres, cline+xres)
        #plt.plot(cwave,cflux,drawstyle='steps')
        #plt.axvline(x=cline, color='r', linestyle='--')
        #plt.show()
        if (cflux_max > cflux_min) & (cwave[cflux_idx_max] <= cline+xres) & (cwave[cflux_idx_max] >= cline-xres):
            if isinstance(contaminating_lines, pd.DataFrame):
                # Could be a real line but avoid all contaminating lines
                if all(np.abs(cwave[cflux_idx_max]-iline) > xres for iline in contaminating_lines['line_wave_um']):
                    lines_dispositions.loc[ii, 'disposition'] = 1
                    lines_dispositions.loc[ii, 'max_amplitude'] = cflux_max-cflux_min
                    lines_dispositions.loc[ii, 'line_flux'] = yflux[idx_cline]
                    potential_lines.append((ii, yflux[idx_cline]))
            else:
                # No need to check for contaminating lines
                lines_dispositions.loc[ii, 'disposition'] = 1
                lines_dispositions.loc[ii, 'max_amplitude'] = cflux_max-cflux_min
                lines_dispositions.loc[ii, 'line_flux'] = yflux[idx_cline]
                potential_lines.append((ii, yflux[idx_cline]))
    if remove_duplicate == 'True':
        # Remove duplicate lines by finding the strongest line in a group of close in wave lines
        # Step 1: Create groups of lines that are within `2 * xres` of each other
        groups = []
        for i, (idx1, _) in enumerate(potential_lines):
            group = [idx1]
            for j, (idx2, _) in enumerate(potential_lines[i+1:], start=i+1):
                if abs(lines_dispositions.loc[idx1, 'line'] - lines_dispositions.loc[idx2, 'line']) <= 2 * xres:
                    group.append(idx2)
                    groups.append(group)
        # Step 2 & 3: Find the strongest line in each group and set the disposition of other lines to 0
        for group in groups:
            if len(group) > 1:
                # Get the indices of lines that are not rare isotopologues
                valid_indices = [idx for idx in group if lines_dispositions.loc[idx, 'species'] == '12C16O']
                # If there are no valid lines, consider all lines in the group
                if not valid_indices:
                    valid_indices = group
                # Find the index of the strongest line in the group among the valid lines
                strongest_line_idx = max(valid_indices, key=lambda idx: lines_dispositions.loc[idx, 'line_flux'])
                # Set disposition of all other lines in the group to 0
                for idx in group:
                    if idx != strongest_line_idx:
                        lines_dispositions.loc[idx, 'disposition'] = 0
    return lines_dispositions

In [7]:
#
def many_Gaussians_list(xl, a, b, *params):
    '''
    Calculates the y-values for a list of x-values for a sum of Gaussian curves with a continuum slope

    Parameters
    ----------
    xl : numpy array
        A listlike object containing x values to evaluate the Gaussian curves at
    a : float
        The constant value of the continuum slope (a in a+b*x)
    b : float
        The slope of the continuum slope (b in a+b*x)
    *params : floats
        Parameters for an arbitrary number of Gaussians, organized such that the 0th parameter is the mean/median/mode of the 0th Gaussian,
        the 1st parameter is the amplitude of the 0th Gaussian, and the 2nd parameter is the standard deviation of the 0th Gaussian, repeating for
        each subsequent Gaussian

    Returns
    -------
    retval : list
        A listlike object containing the y values corresponding to the x values for the given series of Gaussian curves and continuum slope
    '''
    N = 3 # Number of parameters for each Gaussian
    params = np.asarray(params)
    min_freq = min(xl)
    retval = [sum(Gaussian(x, params[i], params[i+1], params[i+2]) for i in np.arange(len(params)//N)*N)+a+b*(x-min_freq) for x in xl]
    return retval

In [8]:
#
# Fit every line with a Gaussian + continuum
def fit_Glines(xwave,yflux,list_wboundaries,detCO_lines,detContaminating_lines,sigma,check='False'):
    '''
    Fit simultaneously many Gaussians and the underlying continuum in given
    chuncks of wavelenghts (given in list_wboundaries)
    Returns:
        1. the sum of the areas of the CO Gaussians (sum_CO_area)
        2. the sum of the areas of the contaminating lines Gaussians (sum_contaminating_area)
        3. the mean of the continuum (mean_cont)
        4. a pandas dataframe with the residual which is good to check the results (df_residual)
    '''
    sum_CO_area, sum_contaminating_area, mean_cont = 0, 0, 0
    df_residual = pd.DataFrame({
        'wave': xwave,
        'residual': [0]*len(xwave)
        })
    print('===== Start fitting...')
    start_time = time.time()
    for ii in range(len(list_wboundaries)-1):
        xwave_min, xwave_max = list_wboundaries[ii], list_wboundaries[ii+1]
        idx_xwave  = (xwave >= xwave_min) & (xwave <= xwave_max)
        print('> Fitting wave range ',np.around(xwave_min,decimals=1),np.around(xwave_max,decimals=1))
        # Filtering the DataFrame based on the disposition and the wavelength range: CO and contaminating lines
        filtered_COlines = detCO_lines[
            (detCO_lines['disposition'] == 1) &
            (detCO_lines['line'] >= xwave_min) &
            (detCO_lines['line'] <= xwave_max)
        ]
        filtered_contaminatinglines = detContaminating_lines[
            (detContaminating_lines['disposition'] == 1) &
            (detContaminating_lines['line'] >= xwave_min) &
            (detContaminating_lines['line'] <= xwave_max)
        ]
        # Find number of contaminating lines for Gauss fit and store
        ncont_lines = filtered_contaminatinglines.shape[0]
        # Append contaminating lines at the end of the CO lines
        filtered_lines = pd.concat([filtered_COlines, filtered_contaminatinglines], ignore_index=True)
        # Create the NumPy array with input params
        p0_2d = np.column_stack((
            filtered_lines['max_amplitude'].astype(float),
            filtered_lines['line'],
            [sigma] * len(filtered_lines)
            ))
        p0 = np.ravel(p0_2d) # 1d array
        p0base, p0slope = np.min(yflux), (yflux[idx_xwave][-1]-yflux[idx_xwave][0])/(xwave[idx_xwave][-1]-xwave[idx_xwave][0])
        p0 = np.insert(p0, [0,0], [p0base,p0slope])
        # Create NumPy arrays with lower and upper bounds
        lower_bounds_2d = np.column_stack((
            [0.0]* len(filtered_lines),
            filtered_lines['line']-sigma,
            [0.5*sigma] * len(filtered_lines)
            ))
        lower_bounds = np.ravel(lower_bounds_2d)
        lower_bounds = np.insert(lower_bounds, [0,0], [p0base-2*np.abs(p0slope),-2*np.abs(p0slope)])
        upper_bounds_2d = np.column_stack((
            filtered_lines['max_amplitude'].astype(float)*1.1,
            filtered_lines['line']+sigma,
            [2*sigma] * len(filtered_lines)
            ))
        upper_bounds = np.ravel(upper_bounds_2d)
        upper_bounds = np.insert(upper_bounds, [0,0], [p0base+2*np.abs(p0slope),2*np.abs(p0slope)]) ## # TODO:
        par, cov = curve_fit(many_Gaussians_list,xwave[idx_xwave],yflux[idx_xwave],p0=p0,bounds=((lower_bounds,upper_bounds)),method='trf',ftol=1e-5,maxfev=10000) # Performs the fit
        a,b = par[0], par[1] # Extracts which parameters relate to the continuum
        par = par[2:] # Extracts which parameters relate to the Gaussians
        try:
            bfit = many_Gaussians_list(xwave[idx_xwave],a,b,*par)
            df_residual.loc[idx_xwave,'residual'] = yflux[idx_xwave]-bfit
            if check == 'True':
                plt.plot(xwave[idx_xwave],yflux[idx_xwave],drawstyle='steps', color='steelblue')
                plt.plot(xwave[idx_xwave],bfit,drawstyle='steps',linestyle='--',color='orange') # Plot the overall fit
                plt.plot(xwave[idx_xwave],yflux[idx_xwave]-bfit,color='grey') # plot the residual
            # Different approach in case there are contaminating lines
            if ncont_lines > 0:
                current_sum_CO = sum(par[jj] * par[jj+2] * np.sqrt(2*np.pi) for jj in range(0, len(par)-3*ncont_lines, 3))
                current_sum_contaminating = sum(par[jj] * par[jj+2] * np.sqrt(2*np.pi) for jj in range(len(par) - 3*ncont_lines, len(par), 3))
                if check == 'True':
                    plt.plot(xwave[idx_xwave],many_Gaussians_list(xwave[idx_xwave],a,b,*par[-3*ncont_lines:]),drawstyle='steps',linestyle='--',color='blue')
            else:
                current_sum_contaminating = 0 # there are no contaminating lines in this wave range!
                current_sum_CO = sum(par[jj] * par[jj+2] * np.sqrt(2*np.pi) for jj in range(0, len(par)-3*ncont_lines, 3))
            if check == 'True':
                plt.savefig('Tests/fig'+str(list_wboundaries[ii])+'.png',dpi=400,bbox_inches='tight')
                plt.draw()
                plt.pause(1)
                plt.close()
        except RuntimeError as e:
            print("curve_fit did not converge! Fit a straight line and assign 3sigma upper limit", e)
            continuum, std_residuals = cont_fit_ransac(xwave[idx_xwave],yflux[idx_xwave],input['deg_polyfit'],show='True')
            current_sum_contaminating = 3*np.median(std_residuals)*sigma*np.sqrt(2*np.pi)
            current_sum_CO = 3*np.median(std_residuals)*sigma*np.sqrt(2*np.pi)
            a = np.nanmedian(continuum)
        #
        sum_CO_area = sum_CO_area + current_sum_CO
        sum_contaminating_area = sum_contaminating_area + current_sum_contaminating
        mean_cont = mean_cont+a
    end_time = time.time()
    print(f"Fitting took {np.around(end_time - start_time,decimals=1)} seconds")
    print('===== End fitting')
    mean_cont = mean_cont/(len(list_wboundaries)-1)
    return sum_CO_area, sum_contaminating_area, mean_cont, df_residual

In [9]:
# Plot data and bestfit model and line centroid if available
def plot_bfit(xwave,yobsflux,ybfitflux,line_peak=False,title=''):
    plt.plot(xwave,yobsflux,color='steelblue',drawstyle='steps')
    plt.plot(xwave,ybfitflux,color='magenta',drawstyle='steps')
    if line_peak:
        plt.axvline(x=line_peak, color='black', linestyle='--')
    if len(title) != 0:
        plt.title(title)
    plt.show()
    return 'Nice plot'

In [10]:
#
# Define a function to fit the cavity width as in eq. 3 of Habel+2021
def cavity_func(x, A, n, B): # but allow for y!=0 when x=0 with extra term B
    return A * np.abs(x)**n + B

In [11]:
#
# Give a set of pixels in SAO ds9 (x,y) [remember python counts from 0 not 1 and x and y are flipped in cube]
# and calculate an average spk. Check where pixels are located in the EWmap
def mean_spk(ewmap,cube,pixels,check='False'):
    plt.imshow(ewmap,origin='lower')
    pix_array = np.array(pixels)
    c_meanspk = np.empty((len(pixels), cube.shape[0]))
    for ii in range(len(pixels)):
        plt.scatter(pix_array[ii][0]-1,pix_array[ii][1]-1,marker='x',color='red',s=50)
        # Extract from cube the spk at said pixel
        c_meanspk[ii] = cube[:, pix_array[ii][1] - 1, pix_array[ii][0] - 1]
    plt.title('EW map with pixels for spectral extraction ')
    plt.show()
    #- Calculate mean spk and show if for checking purposes
    meanspk = np.nanmean(c_meanspk,axis=0)
    if check == 'True':
        plt.plot(wave[inwave], c_meanspk.T[inwave],drawstyle='steps',label=pixels)
        plt.xlabel('Wavelength')
        plt.ylabel('Flux')
        plt.title('Extracted Spectra with mean in black')
        plt.plot(wave[inwave],meanspk[inwave],color='black',drawstyle='steps')
        plt.legend()
        plt.show()
    return meanspk

In [12]:
#
# Find the lowest possible pixel in a given box in an image
def find_lowest_positive_pixel(image, top_left, bottom_right):
    # Convert the image to a NumPy array
    image_array = np.array(image)
    # Extract the specified box from the image array
    box = image_array[top_left[0]:bottom_right[0]+1, top_left[1]:bottom_right[1]+1]
    # Mask the non-positive values within the box
    positive_mask = (box > 0)
    # Apply the mask to the box
    positive_pixels = box[positive_mask]
    if len(positive_pixels) == 0:
        return None
    # Find the index of the pixel with the lowest positive value within the box
    min_pixel_index = np.nanargmin(positive_pixels)
    # Find the coordinates of the pixel with the lowest positive value within the box
    min_pixel_coords = np.argwhere(positive_mask)[min_pixel_index]
    # Adjust the coordinates to be relative to the entire image
    min_pixel_coords[0] += top_left[0]
    min_pixel_coords[1] += top_left[1]
    #
    # Show the image in log scale (otherwise range not visible NOW values will be negative)
    # and draw the box on the image
    #plt.imshow(np.log10(image))#,origin='lower')
    plt.imshow(image,vmin=0,vmax=2)
    rect = plt.Rectangle((top_left[1], top_left[0]), bottom_right[1] - top_left[1], bottom_right[0] - top_left[0],
                         edgecolor='g', linewidth=2, facecolor='none')
    #width, height = bottom_right[0]-top_left[0], top_left[1]-bottom_right[1]
    #rect = plt.Rectangle((top_left[0], top_left[1]-height), width, height, edgecolor='g', linewidth=2, facecolor='none')
    plt.gca().add_patch(rect)
    plt.scatter(min_pixel_coords[1], min_pixel_coords[0], marker='x', color='red', s=100)
    print('>>> Min pixel x,y from EW map for ds9(+1) (REF pixel) ', min_pixel_coords[1]+1,min_pixel_coords[0]+1)
    plt.show()
    #
    return tuple(min_pixel_coords)

In [13]:
#
# Function for Savitzky-Golay filtering
def savitzky_golay_smoothing(data, window_size, order):
    return savgol_filter(data, window_size, order)
#

In [14]:
# Function to resample data
def resamp(x,y,num):
    new_x = np.linspace(x.min(), x.max(), num=num)  # Generate new x-coordinates
    new_y = np.interp(new_x, x, y)  # Perform linear interpolation on y-coordinates
    return [new_x,new_y]

In [15]:
#
# Function to find the x-indices to fit a continuum in a y-spectrum full of lines
def indx_cont(x,y,sigma_clip):
    mean, median, std = sigma_clipped_stats(y, sigma=sigma_clip, maxiters=10)
    indx = np.where((y >= median-std) & (y <= median+std))[0]
    return indx

In [16]:
#
# Function to fit the continuum in a subset of the wavelegths for alreday filtered data
def cont_fit_ransac(x,y,pdeg,show=False,return_params=False,exclude_wavelength_ranges=None):
    # Combine x and y into a single array
    data = np.column_stack((x, y))
    # Remove rows with NaN values for fitting
    valid_data = data[~np.isnan(data).any(axis=1)]
    # Separate the input variables and target variable for fitting
    xfit = valid_data[:, 0]
    yfit = valid_data[:, 1]
    if exclude_wavelength_ranges is not None:
        # Filter data to exclude points within the specified wavelength ranges
        for exclude_range in exclude_wavelength_ranges:
            mask = ~((xfit >= exclude_range[0]) & (xfit <= exclude_range[1]))
            xfit = xfit[mask]
            yfit = yfit[mask]

    # Create a pipeline with polynomial feature transformation and RANSACRegressor
    ransac_pipeline = make_pipeline(
        PolynomialFeatures(degree=pdeg),
        RANSACRegressor()
        )
    # Fit the pipeline on the valid data and calculate the stddev of the residulas on the valid data [at the moment I do not divide by sqrt(points)]
    ransac_pipeline.fit(xfit[:, np.newaxis], yfit)
    std_residuals = np.std(yfit - ransac_pipeline.predict(xfit[:, np.newaxis]))#/np.sqrt(len(xfit))
    # Predict y-values for all x-values (including NaNs)
    yfit_all = ransac_pipeline.predict(x[:, np.newaxis])
    # Best fit parameters
    best_fit_params = ransac_pipeline.named_steps['ransacregressor'].estimator_.coef_
    # Replace predicted values corresponding to NaN x-values with NaN
    yfit_all[np.isnan(x)] = np.nan
    #
    if show=='True':
        plt.plot(x,y,color='steelblue',drawstyle='steps')
        plt.plot(x,yfit_all,color='orange')
        plt.ylim(np.nanmin(y),np.nanmax(y))
        plt.xlim(np.nanmin(x),np.nanmax(x))
        plt.xlabel('Wavelength [micron]')
        plt.show()
    if return_params:
        return best_fit_params,std_residuals
    else:
        return yfit_all,std_residuals

In [17]:
#
# Function to find the mean value in a valley and
# the minimum between the two max peaks of a curve
def mid_valley(data,window_size,order,show='False'):
    smoothed_ydata = savgol_filter(data,window_size,order)
    ydata = data
    xdata = np.linspace(0,len(ydata))
    # Find the main two peaks
    prominence, step = 1, 0.1
    peaks = []
    while len(peaks) != 2 and prominence > 0:
        peaks, _ = find_peaks(ydata, prominence=prominence)
        prominence = prominence - step
    main_peaks = peaks[np.argsort(ydata[peaks])[-2:]]
    main_peaks.sort()
    # Valleys with width half the main peaks
    valleys, valleys_properties = find_peaks(-ydata, prominence=1, width=(main_peaks[1]-main_peaks[0])/2.)
    if len(valleys) > 0:
        widths, width_heights, left_ips, right_ips = peak_widths(-ydata, valleys, rel_height=0.5)
        # Find the largest plateau based on the width
        largest_plateau_index = np.argmax(widths)
        left_ip = left_ips[largest_plateau_index]
        right_ip = right_ips[largest_plateau_index]
        # Calculate the midpoint of the largest flat valley
        midpoint = (left_ip + right_ip) / 2
    else:
        # There is no plateu so I get the minimum between the peaks
        min_index = np.argmin(ydata[main_peaks[0]:main_peaks[1]])
        midpoint = min_index+main_peaks[0]
    if show == 'True':
        plt.plot(data,color='grey',label='original')
        plt.plot(ydata,color='black',label='smoothed')
        plt.axvline(midpoint,color='black',linestyle=':')
        plt.axvline(main_peaks[0],color='black',linestyle=':')
        plt.axvline(main_peaks[1],color='black',linestyle=':')
        plt.legend()
        plt.show()
    return round(midpoint), midpoint

In [18]:
#
# Read+rotate image and plot before and after rotation
def rot_ima(hdul,rot_angle_deg,fits2save='',show='False'):
    # check for NANs and replace with zeros
    hdul.data = np.nan_to_num(hdul.data, nan=0)
    # rotate image
    rotated = rotate(hdul.data,rot_angle_deg,reshape=False)
    angle_ori_rad = np.arccos(-hdul.header['PC1_1']) # = to np.arcsin(-hdul.header['PC1_2'])
    if len(fits2save) != 0: # save the rotated image in a fits file
        new_header = fits.Header()
        new_header['ROT_DEG'] = (str(rot_angle_deg), 'Angle rotatation clockwise deg') #from vertical by which image has been rotated
        hdul.header.extend(new_header, update=True)
        #rot_angle_rad = np.deg2rad(rot_angle_deg) # Angle to rotate in radians
        #hdul.header['PC1_1'] = -np.cos(rot_angle_rad+angle_ori_rad)
        #hdul.header['PC1_2'] = -np.sin(rot_angle_rad+angle_ori_rad)
        #hdul.header['PC2_1'] = -np.sin(rot_angle_rad+angle_ori_rad)
        #hdul.header['PC2_2'] = np.cos(rot_angle_rad+angle_ori_rad)
        out_fits = fits.PrimaryHDU(data=rotated,header=hdul.header)
        out_fits.writeto(fits2save, overwrite=True)
    if show == 'True':
        f, axarr = plt.subplots(1,2)
        axarr[0].imshow(hdul.data,origin='lower')
        axarr[1].imshow(rotated,origin='lower')
        plt.show()
    return rotated

In [19]:
# Extract the wavelength array from a datacube, indx which to read
def get_wave(hdul,indx):
    header_cube = hdul[indx].header
    cube=hdul[indx].data
    # get wave info and define grid in micron (header in AA), redshift is zero for us
    nz, ny, nx = cube.shape
    crdelt3 = header_cube['CDELT3']
    crval3 = header_cube['CRVAL3']
    wave =((crdelt3 * (np.arange(0,nz,1))) + crval3)
    return wave

In [20]:
#
# Script to find the angle of the jet with respect to vertical
def jet_angle(filein,show='False'):
    hdu = fits.open(filein)[0] #jet image
    # for each x smooth the y-array and find the peak
    smoothed_data = gaussian_filter1d(hdu.data, sigma=3, axis=0)
    loc_peak = smoothed_data.argmax(axis=0)
    x, y = np.arange(loc_peak.size), loc_peak
    # Initialize the RANSAC regressor and fit the peaks vs x to find the slope
    ransac = RANSACRegressor()
    ransac.fit(x.reshape(-1,1),y)
    print('>>> Ransac estimator coeff.  ', ransac.estimator_.coef_[0])
    if ransac.estimator_.coef_[0] > 0: # because rotation will be done clockwise
        angle = -(90. - np.rad2deg(np.arctan(ransac.estimator_.coef_[0])))
    else:
        angle = 90. + np.rad2deg(np.arctan(ransac.estimator_.coef_[0]))
    if show == 'True':
        plt.plot(x,y, 'o', color='lightblue')
        plt.plot(x,ransac.estimator_.intercept_+ransac.estimator_.coef_[0]*x,color='red')
        plt.title('Jet angle(deg) w respect to vertical '+str(angle))
        plt.show()
    return angle

In [21]:
#
# Horizontal cuts to measure the FWHM (in arcsec) of the jet: jet needs to be vertically aligned
def jet_xyfwhm(rotated,xloc,pix_arcsec):
    # xloc is the xposition of the jet max emission
    xwidth=10 # on each size of the peak to measure width wih Gauss fit
    xdata = np.arange(xloc-xwidth, xloc+xwidth+1)
    std_xloc = np.std(rotated[:,xloc])
    xjet, yjet, fwhm = [], [], []
    for i in range(rotated.shape[0]):
        ydata = rotated[i,xloc-xwidth:xloc+xwidth+1]
        # first fit and remove the continuum and scale peak to 1
        # the latter helps with the fit
        continuum, std_residuals = cont_fit_ransac(xdata,ydata,1,show='False')
        ydata = (ydata-continuum)/np.nanmax(ydata)
        try:
            # fit a Gaussian
            par, cov = curve_fit(Gaussian,xdata,ydata,  # p0=(1,xwidth,0.7), \
                        bounds =([0.5,xloc-3,0.3],
                                 [1.5,xloc+3,3]),
                                 method='trf', maxfev=100000)
            print(f"Parameters [amp,cen,sig]: {i} {np.around(par[0],decimals=2)} {np.around(par[1],decimals=2)} {np.around(par[2],decimals=2)}")
            fit_y = Gaussian(xdata, par[0],par[1],par[2])
            # Check: is this a good fit?
            plotRes = 'True'
            if (par[0] > 2*np.std(ydata)) and (np.abs(par[1]-xloc) < 4) and (par[2] > 0.4) and (par[2] < 2):
                if plotRes == 'True':
                    plt.plot(xdata,ydata,color='steelblue',drawstyle='steps')
                    plt.axvline(x=xloc, color='black', linestyle=':')
                    plt.plot(xdata,fit_y,color='orange',linestyle='-',drawstyle='steps')
                    plt.axvline(x=par[1], color='orange', linestyle='--')
                    plt.title('y value and xloc '+str(i)+' '+str(xloc))
                    #plt.show()
                    plt.pause(0.2)
                    plt.clf()  # Clears the current figure and prepares it for the next plot
                xjet.append(par[1])
                yjet.append(i)
                fwhm.append(2.35*par[2]* pix_arcsec) # in arcsec while other vals are in pixels
            else:
                xjet.append(np.nan)
                yjet.append(np.nan)
                fwhm.append(np.nan)
        except RuntimeError as e:
            xjet.append(np.nan)
            yjet.append(np.nan)
            fwhm.append(np.nan)

    val2return = [list(row) for row in zip(xjet,yjet,fwhm)]
    return val2return

In [22]:
#
# Horizontal cuts to measure the FWHM (in arcsec) of the jet: jet needs to be vertically aligned
def jet_xyfwhmwunc(rotated,xloc,pix_arcsec):
    #---
    # Find the location of the jet, its FWHM, and the unc on the centroid
    # using the Porter relation: rms = 0.4 FWHM/SNR which
    # is valid as long as we are Nyquist sampled, ie FWHM > 2 pixels
    #---
    # xloc is the xposition of the jet max emission
    xwidth=10 # on each size of the peak to measure width wih Gauss fit
    xdata = np.arange(xloc-xwidth, xloc+xwidth+1)
    std_xloc = np.std(rotated[:,xloc])
    xjet, yjet, fwhm, rms_fwhm = [], [], [], []
    for i in range(rotated.shape[0]):
        ydata = rotated[i,xloc-xwidth:xloc+xwidth+1]
        # first fit and remove the continuum and scale peak to 1
        # the latter helps with the fit
        continuum, std_residuals = cont_fit_ransac(xdata,ydata,1,show='False')
        ydata = (ydata-continuum)/np.nanmax(ydata)
        try:
            # fit a Gaussian
            par, cov = curve_fit(Gaussian,xdata,ydata,  # p0=(1,xwidth,0.7), \
                        bounds =([0.5,xloc-3,0.3],
                                 [1.5,xloc+3,3]),
                                 method='trf', maxfev=100000)
            print(f"Parameters [amp,cen,sig]: {i} {np.around(par[0],decimals=2)} {np.around(par[1],decimals=2)} {np.around(par[2],decimals=2)}")
            fit_y = Gaussian(xdata, par[0],par[1],par[2])
            # find regions outside the FWHM where to estimate the noise
            lower_bound = par[1] - (2.35 * par[2]) # 2*HWHM otherwise I get points in the line to estimate rms
            upper_bound = par[1] + (2.35 * par[2]) # 2*HWHM otherwise I get points in the line to estimate rms
            xindices4noise = (xdata > np.min(xdata)) & (xdata < lower_bound) | (xdata > upper_bound) & (xdata < np.max(xdata))
            xdata4noise, ydata4noise = xdata[xindices4noise], ydata[xindices4noise]
            snr = par[0]/np.std(ydata4noise)
            # Check: is this a good fit?
            plotRes = 'True'
            if (par[0] > 2*np.std(ydata)) and (np.abs(par[1]-xloc) < 4) and (par[2] > 0.4) and (par[2] < 2):
                if plotRes == 'True':
                    plt.plot(xdata,ydata,color='steelblue',drawstyle='steps')
                    plt.axvline(x=xloc, color='black', linestyle=':')
                    plt.plot(xdata,fit_y,color='orange',linestyle='-',drawstyle='steps')
                    plt.plot(xdata4noise, ydata4noise, 'o', color='blue')
                    plt.title('y value and xloc '+str(i)+' '+str(xloc))
                    #plt.show()
                    plt.pause(0.2)
                    plt.clf()  # Clears the current figure and prepares it for the next plot
                xjet.append(par[1])
                yjet.append(i)
                fwhm.append(2.35*par[2]*pix_arcsec) # in arcsec while other vals are in pixels
                rms_fwhm.append(0.4*2.35*par[2]/snr*pix_arcsec) # in arcsec
            else:
                for lst in [xjet, yjet, fwhm, rms_fwhm]:
                    lst.append(np.nan)
        except RuntimeError as e:
            for lst in [xjet, yjet, fwhm, rms_fwhm]:
                lst.append(np.nan)
    val2return = [list(row) for row in zip(xjet,yjet,fwhm,rms_fwhm)]
    return val2return

In [ ]:
#
# Routine to measure the wind edges: yxloc gives the center defined from the jet
def auto_wind_edges(rotated,yxloc,xwidth,fracpeak):
    #
    smoothed = gaussian_filter1d(rotated[:,yxloc[1]],3)
    # fracpeak: fraction of peak given in Input_radcutwidth
    threshold = fracpeak*np.nanmax(smoothed)
    ny, nx = rotated.shape
    y_indices  = np.where(rotated[yxloc[0]:ny,yxloc[1]] >= threshold)[0]
    x = np.arange(yxloc[1]-xwidth,yxloc[1]+xwidth)
    roots = []
    for i in y_indices:
        selected_x = []  # Initialize selected_x within the loop
        y = rotated[yxloc[0]+i,yxloc[1]-xwidth:yxloc[1]+xwidth]
        ys =  gaussian_filter1d(y,1) #smoothed curve: for broad curves can choose 2 but for narrow one needs 1
        y1, y2 = np.gradient(ys, x), np.gradient(np.gradient(ys, x), x) #first & second derivatives
        interpolation = CubicSpline(x, y2)
        dummy_roots = interpolation.roots()
        # There should be only 2 inflection points delineating the right and left edges of the wind
        # if there are more than 2, take the two closet to the midpoint
        if len(dummy_roots) > 2 :
            arr = np.abs(dummy_roots-yxloc[1])
            closest_indxs = sorted(range(len(arr)), key=lambda i: arr[i])[:2]
            xroots = dummy_roots[closest_indxs]
            # Added to solve issue for FSTauB and TAU042021 [but does not work for HH30]
            # the next edge location should be larger than the previous
            # Uncomment for FSTauB and TAU042021 (but comment for HH30)
            if (i > 0) & (len(roots) > 0):
                if (xroots[1] < roots[-1][0]):
                    arr[closest_indxs] = float('inf')
                    second_closest_indxs = sorted(range(len(arr)), key=lambda i: arr[i])[:2] #np.argmin(arr)
                    xroots = dummy_roots[second_closest_indxs]
        else:
            xroots = dummy_roots
        #
        # ALWAYS check the determination of inflection points per y slice
        # IF manual == 'True' inflection points can be manually selected
        fig = plt.figure()
        plt.plot(x,y,color='grey')
        plt.plot(x,ys,'o',color='black')
        plt.plot(x,y2,'o',color='magenta')
        plt.plot(x,interpolation(x),color='pink')
        plt.axhline(0,color='blue')
        for k in range(len(xroots)):
            plt.axvline(xroots[k],linestyle='--',color='black')
        plt.title('y point '+str(yxloc[0]+i))
        plt.show()
        yroots = np.full(len(xroots), yxloc[0]+i)
        roots.extend(list(zip(xroots, yroots))) # list of tuples
    return roots

In [ ]:
#
# Routine to superimpose the wind edges onto the wind image
def wedges_on_ima(ima,wedges,xyfwhm_jet=None,star_loc=None,png2save=''):
    #ima is the rotated image with wind cavity vertical
    xp, yp = zip(*wedges) #unpack list of tuples
    plt.imshow(np.sqrt(ima),origin='lower')
    plt.plot(xp,yp,'o',color='white',markerfacecolor='none')
    if xyfwhm_jet is not None:
        xjet, yjet, fwhm = zip(*xyfwhm_jet)
        plt.plot(xjet, yjet, '+-', color='white')
    if star_loc is not None:
        plt.plot(star_loc[0], star_loc[1], 'x', color='yellow', markersize=10, markeredgewidth=2)
        plt.axvline(x=star_loc[0], color='yellow', linestyle='--')
    if len(png2save) != 0:
        plt.savefig(png2save)
    plt.show()
    new_roots = 0
    return new_roots

In [ ]:
#
# Routine to measure the wind opening angle using eq. 3 of Habel+2021
# input x and y are in arcsec with respect to center
def wind_opening(x,y,Rmax,jet_location=None,png2save=''):
    # x and y are the inflection points in arcsec with respect to center
    # Rmax: outermost edged of the wind in arcsec
    #
    #- Fit all the data with a power law that does not necessarily pass through y=0 for x=0
    #  using curvefit
    popt, pcov = curve_fit(cavity_func, x, y, maxfev=5000)
    A_opt, n_opt, B_opt = popt
    theta_opt = np.arctan((Rmax**(1-n_opt)/A_opt)**(1/n_opt))
    if theta_opt < 0: #to have angle between 0 and 2pi
        theta_opt += np.pi
    print('=== Curve-fit results [extended power law A, n, B]: ', np.around(A_opt,decimals=2), np.around(n_opt,decimals=2), np.around(B_opt,decimals=2))
    print('  Pow_order, Semi-open_angle [deg] ', np.around(n_opt,decimals=3), np.around(np.degrees(theta_opt),decimals=3))
    print('  Footpoint in arcsec (where y=0) ', np.around((np.abs(B_opt/A_opt))**(1./n_opt),decimals=3))
    #- Find the midpoints but fit separately positive and negative edges with a straight line
    # using linear regression with RANSAC
    #xmid = np.mean(x.reshape(-1, 2), axis=1)
    #ymid = np.mean(y.reshape(-1, 2), axis=1)
    xneg, xpos = x[(x < 0)], x[(x > 0)]
    yneg, ypos = y[(x < 0)], y[(x > 0)]
    neg_ransac, pos_ransac = linear_model.RANSACRegressor(), linear_model.RANSACRegressor()
    neg_ransac.fit(xneg[:,np.newaxis],yneg)
    pos_ransac.fit(xpos[:,np.newaxis],ypos)
    print('=== Ransac Linear Fit xneg,xpos: ', \
        np.around(neg_ransac.estimator_.coef_[0],decimals=2),np.around(neg_ransac.estimator_.intercept_,decimals=2), \
        np.around(pos_ransac.estimator_.coef_[0],decimals=2),np.around(pos_ransac.estimator_.intercept_,decimals=2))
    print('>>> Semi-open_angle xneg,xpos [deg] ', \
        np.around(90+np.degrees(np.arctan(neg_ransac.estimator_.coef_[0])),decimals=2), np.around(90-np.degrees(np.arctan(pos_ransac.estimator_.coef_[0])),decimals=2))
    print('  Footpoint in arcsec xneg,xpos (where y=0) ', \
        np.around(-neg_ransac.estimator_.intercept_/neg_ransac.estimator_.coef_[0],decimals=2), np.around(-pos_ransac.estimator_.intercept_/pos_ransac.estimator_.coef_[0],decimals=2))
    #- Plotting
    xgrid = np.linspace(np.min(x),np.max(x),100)
    plt.plot(xgrid,cavity_func(xgrid,A_opt,n_opt,B_opt),'-',color='grey')
    plt.plot(x,y,'o',color='lightblue')
    plt.plot(xgrid,neg_ransac.predict(xgrid[:,np.newaxis]),'-.',color='black')
    plt.plot(xgrid,pos_ransac.predict(xgrid[:,np.newaxis]),':',color='black')
    #plt.plot(xmid, ymid,'o',color='black')
    if jet_location != None:
        plt.plot(jet_location[0],jet_location[1],'+',markersize=10,color='green',markeredgewidth=2)
        plt.text(0.05, 0.95, 'jet', transform=plt.gca().transAxes,fontsize=12,color='green',verticalalignment='top',fontweight='bold')
    plt.xlabel('Radial Distance [arcsec]')
    plt.ylabel('Vertical Height above the Disk [arcsec]')
    plt.ylim(-0.2,np.nanmax(y)*1.5)
    if len(png2save) != 0:
        plt.savefig(png2save)
    plt.show()
    return A_opt, n_opt, B_opt
#